In [ ]:

!mkdir -p cache

import os
import time

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

dataset = load_dataset("Abirate/english_quotes", split="train")
sample_size = max(1, int(dataset.num_rows * 0.1))
data = dataset.shuffle(seed=42).select(range(sample_size))  # Sample 10%
data = data.map(
    lambda samples: tokenizer(
        samples["quote"],
        padding="longest",
        truncation=True,
        max_length=128,
    ),
    batched=True,
)
train_sample = data.select(range(5))
display(train_sample)

from peft import LoraConfig, get_peft_model

# Fill in `r=1` and `target_modules`.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,  # a scaling factor that adjusts the magnitude of the weight matrix. Usually set to 1
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",  # this specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM",
)

# Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

# Fill out the `Trainer` class.
output_directory = os.path.join("../cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,  # Higher learning rate than full fine-tuning.
    num_train_epochs=3,
    per_device_train_batch_size=1,
    use_cpu=True,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# Save the actual PEFT model and tokenizer
peft_model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

# Generate output tokens
peft_model.eval()
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
with torch.no_grad():
    outputs = peft_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50,
        do_sample=True,
        top_k=50,
        top_p=0.95,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

IndentationError: unexpected indent (27900670.py, line 54)